# Lab 6 — RAG, step by step (interactive)

Open this as a **notebook** (`explore.ipynb`) or run the `.py` cell-by-cell in VSCode with
**Shift+Enter** — same content either way. It walks retrieval one stage at a time:
corpus → chunks → embeddings → vector store → retrieve → grounded answer.
**Fully offline** (local hashing embeddings) — no API key or network needed.

This drives the same code as `rag.py`; the command line is `python rag.py "your question"`.

**How to work through it** — three beats:
- 🔮 **Predict** — before you run a cell, say what you expect. Then run and compare.
- 🛠️ **Your turn** — one cell is yours to change; the reference answer is in the README Playbook.
- 🗣️ **Explain** — end by answering the question in your own words.

In [1]:
import os
import sys

# Make the lab's modules importable whether cells run from the repo root or here.
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
sys.path.insert(0, HERE)

import numpy as np  # noqa: E402
from rag import (  # noqa: E402
    load_corpus, chunk_text, get_embedder, build_index, retrieve,
    ground_answer, CORPUS_DIR, MIN_SCORE,
)

## 1. The corpus and how it chunks

In [2]:
docs = load_corpus(CORPUS_DIR)
print(f"{len(docs)} documents:", [name for name, _ in docs])
chunks = chunk_text(docs[0][1])
print(f"\n{docs[0][0]} -> {len(chunks)} chunks. First chunk:\n")
print(chunks[0])

4 documents: ['accounts.md', 'returns.md', 'shipping.md', 'warranty.md']

accounts.md -> 1 chunks. First chunk:

# Northwind Accounts & Privacy
## Creating an account
You can check out as a guest, but an account lets you track orders and save addresses. We never
sell your personal data. You can request deletion of your account at any time.
## Password reset
Use the "Forgot password" link on the sign-in page. A reset link is valid for **one hour**. If
it expires, request a new one.
## Loyalty program
Members earn 1 point per dollar spent. 100 points convert to a $5 reward. Points expire after
12 months of account inactivity.


## 2. Build the vector index (embed every chunk)

In [3]:
embedder, used = get_embedder("local")
store = build_index(embedder)
print(f"embeddings: {used} | chunks indexed: {len(store.meta)} | "
      f"vector dim: {store.matrix.shape[1]}")

embeddings: local | chunks indexed: 5 | vector dim: 512


## 3. Retrieve for an on-topic question

🔮 **Predict first:** the question below is *"how long do I have to return something?"*
Which document do you think the top passage will come from — `returns.md`, `shipping.md`,
or `accounts.md`? And will the top score be large (~0.5) or modest (~0.1)? Run and check — the
number is smaller than people expect, because this toy embedder only counts shared words.

In [4]:
hits = retrieve(store, embedder, "how long do I have to return something?", k=4)
for s, m in hits:
    print(f"[{s:.3f}] {m['source']}#chunk{m['chunk_id']}: "
          f"{' '.join(m['text'].split())[:80]}...")

[0.140] returns.md#chunk1: If an item arrives damaged or you received the wrong product, contact support wi...
[0.132] returns.md#chunk0: # Northwind Returns & Refunds Policy ## Return window You may return most items ...
[0.000] accounts.md#chunk0: # Northwind Accounts & Privacy ## Creating an account You can check out as a gue...
[0.000] shipping.md#chunk0: # Northwind Shipping Policy ## Delivery times Standard shipping arrives in **3 t...


## 4. The off-topic guard — scores collapse toward 0

🔮 **Predict first:** *"what is the capital of France?"* shares almost no content words with a
returns-and-shipping corpus. Will its top score clear `MIN_SCORE`, or fall below it? What should
the agent answer when nothing clears the bar?

In [5]:
off = retrieve(store, embedder, "what is the capital of France?", k=4)
print("top score:", round(off[0][0], 3), " | MIN_SCORE:", MIN_SCORE)
answer, _ = ground_answer("what is the capital of France?", off)
print(answer)

top score: 0.057  | MIN_SCORE: 0.08
I couldn't find that in the knowledge base, so I won't guess. A human should follow up.


## 5. Grounded answer + citations for the on-topic question

In [6]:
answer, cites = ground_answer("how long do I have to return something?", hits)
print(answer)
print("\nCitations:", cites)

Based on the knowledge base:

- If an item arrives damaged or you received the wrong product, contact support within 7 days and
we will send a prepaid return label and a replacement at no cost.

- # Northwind Returns & Refunds Policy
## Return window
You may return most items within **30 days** of delivery for a full refund. Items marked
"final sale" cannot be returned. Opened software and gift cards are non-returnable.
## How refunds are issued
Refunds go back to the original payment method within **5 business days** after we receive the
returned item. If the original card is closed, we issue store credit.
## Damaged or wrong items

(Grounded in the retrieved passages above - verify against the sources.)

Citations: ['returns.md#chunk1', 'returns.md#chunk0', 'accounts.md#chunk0', 'shipping.md#chunk0']


## 6. 🛠️ Your turn — find the abstain threshold

The anti-hallucination gate lives in **one number**: a question whose top score is below
`THRESHOLD` gets *"not in the knowledge base"* instead of a guess. Below are two on-topic and two
off-topic questions. Change `THRESHOLD` and find the value that **answers both on-topic questions
and abstains on both off-topic ones**. Then push it too high and too low to see what breaks.

*Reference answer + discussion in the README Teacher's Playbook.*

In [7]:
THRESHOLD = MIN_SCORE   # 🛠️ try values between 0.04 and 0.25 and watch the line move

probe = [
    ("how long do I have to return something?", "on-topic"),
    ("do you refund shipping?",                 "on-topic"),
    ("what is the capital of France?",          "off-topic"),
    ("do you price match a competitor?",        "off-topic"),
]
for q, label in probe:
    top = retrieve(store, embedder, q, k=4)[0][0]
    decision = "ANSWER " if top >= THRESHOLD else "ABSTAIN"
    print(f"{top:.3f}  {decision}  ({label:<9}) {q}")
print(f"\nthreshold = {THRESHOLD}")

0.140  ANSWER   (on-topic ) how long do I have to return something?
0.454  ANSWER   (on-topic ) do you refund shipping?
0.057  ABSTAIN  (off-topic) what is the capital of France?
0.000  ABSTAIN  (off-topic) do you price match a competitor?

threshold = 0.08


## 7. 🗣️ Explain

In one or two sentences: **why** does *"capital of France"* score near zero while
*"return policy"* scores high, even though this toy embedder never learned any language?
(Hint: look at `_tokens` and `_STOPWORDS` in `rag.py` — what actually drives the similarity?)

---
### (optional) Swap in hosted embeddings and compare
Set `OPENAI_API_KEY`, then uncomment to see how learned embeddings change retrieval:

In [8]:
# os.environ["EMBEDDINGS"] = "openai"
# emb2, used2 = get_embedder("openai")
# store2 = build_index(emb2)
# for s, m in retrieve(store2, emb2, "warranty on electronics?", k=4):
#     print(f"[{s:.3f}] {m['source']}#chunk{m['chunk_id']}")